# **CARE (2D), powered by CAREamics**

---

<font size = 4> CARE (Content-Aware image REstoration) is a **supervised** deep-learning method that restores low-quality microscopy images. It is trained on **pairs of low-SNR images and matching high-SNR (ground truth) images of the same field of view**, and learns the mapping from one to the other. It was originally published by [Weigert *et al.*, Nature Methods (2018)](https://doi.org/10.1038/s41592-018-0216-7).

<font size = 4> **This notebook runs CARE on 2D datasets using [CAREamics](https://careamics.github.io/), a modern PyTorch/Lightning implementation. It is an alternative to the existing CSBDeep-based `CARE_2D_ZeroCostDL4Mic` notebook.**

<font size = 4> **If you do not have high-SNR ground truth**, CARE cannot be used. Consider the **CAREamics Noise2Noise** notebook (two independent noisy acquisitions) or the **CAREamics Noise2Void** notebook (noisy images alone) instead.

---

<font size = 4>*Disclaimer*:

<font size = 4>This notebook is part of the Zero-Cost Deep-Learning to Enhance Microscopy project (https://github.com/HenriquesLab/DeepLearning_Collab/wiki). Jointly developed by the Jacquemet (https://cellmig.org/) and Henriques (https://henriqueslab.github.io/) laboratories.

<font size = 4>You can also use CARE in **CAREamics** (https://github.com/CAREamics/careamics).

<font size = 4>This notebook is based on:

<font size = 4>**Content-aware image restoration: pushing the limits of fluorescence microscopy**, Weigert *et al.*, Nature Methods 2018 (https://doi.org/10.1038/s41592-018-0216-7)

<font size = 4>**Please cite the original CARE paper and CAREamics when using this notebook.**


# **How to use this notebook?**

---

<font size = 4>This notebook is structured in numbered sections. Run the cells from top to bottom.

---
### **Structure of a notebook**

<font size = 4>**Text cells** provide information. **Code cells** contain code; move your cursor over the `[ ]` on the left and click the play button to execute.

---
### **Making changes to the notebook**

<font size = 4>**Make a copy** of this notebook and save it to your Google Drive (`File -> Save a copy in Drive`) before editing.

# **0. Before getting started**
---

<font size = 4>Make sure you are logged into your Google account and that your data is in your Google Drive.

<font size = 4>CARE is **supervised**: for each field of view you need a **low-SNR image** and a matching **high-SNR image** of the same scene. The low-SNR image is the input, the high-SNR image is the target. A second equally-noisy acquisition will not work; use the **CAREamics Noise2Noise** notebook for that.

<font size = 4>Put the two sets in **separate folders**, with **matching file names** across them. Only `.tif` and `.tiff` files are supported. See the [ZeroCostDL4Mic wiki](https://github.com/HenriquesLab/ZeroCostDL4Mic/wiki) for how to prepare a training dataset.

<font size = 4>**We strongly recommend also preparing a Quality Control dataset**: a few low-SNR images with matching high-SNR versions. Section 5.2 uses these to measure how much the model improved your images. **Hold these fields of view back from training**, otherwise 5.2 only measures how well the model fits data it has already seen.

<font size = 4>A common data structure that works well:

*   Data
    - **Training**
        - source (low SNR): img_1.tif, img_2.tif ...
        - target (high SNR, same scenes): img_1.tif, img_2.tif ...
    - **Quality control** (optional but recommended)
        - Low SNR images: img_1.tif, img_2.tif ...
        - High SNR images: img_1.tif, img_2.tif ...
    - **Prediction**: images to restore
    - **Results**

---
<font size = 4>**Important note**

<font size = 4>To **train from scratch**: run **sections 1-4**, then **section 5** to assess quality and **section 6** to predict.
<font size = 4>To **continue training from a checkpoint**: run **sections 1-4** with `Use_pretrained_model` enabled and training paths filled in.
<font size = 4>To only **run predictions with an existing checkpoint**: run **sections 1-3**, enable `Use_pretrained_model`, run **section 4.1** to load the model, skip **section 4.2**, then run **section 6**.
---


# **1. Install CAREamics and dependencies**
---

## **1.1. Install CAREamics**

In [ ]:
#@markdown ##Install CAREamics and dependencies
#@markdown This installs a pinned, tested version of CAREamics. It takes several minutes,
#@markdown because CAREamics 0.3.2 requires torch < 2.10 and Colab ships a newer one, so
#@markdown PyTorch and its CUDA libraries are reinstalled.
!pip install "careamics==0.3.2" "careamics-portfolio" -q

print("CAREamics installed.")

## **1.2. Restart the runtime (only if you see an import error)**
<font size = 4>The install above keeps Colab's existing NumPy, so you can normally continue straight to section 1.3. **If section 1.3 raises a NumPy or import error**, go to `Runtime -> Restart session`, then re-run from section 1.3 (do **not** re-run the install cell).

## **1.3. Load key dependencies**

In [ ]:
#@markdown ##Load key dependencies
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile

import careamics
from careamics import CAREamist
from careamics.config import create_advanced_care_config
from careamics.metrics.metrics import scale_invariant_psnr
from microssim import micro_structural_similarity

TIFF_SUFFIXES = {".tif", ".tiff"}


def require_path(path_value, field_name):
    """Return `path_value` as a Path, with a clear error if the field was left unset.

    An empty #@param becomes `Path(".")`, which exists, so CAREamics would silently
    search the whole Colab working directory instead of reporting the mistake.
    Paths that must already exist are checked by CAREamics itself.
    """
    path_text = str(path_value).strip()
    if not path_text:
        raise ValueError(f"Please set `{field_name}` before running this cell.")
    return Path(path_text).expanduser()


print(f"CAREamics version: {careamics.__version__}")


# **2. Initialise the Colab session**
---

## **2.1. Check for GPU access**

In [ ]:
#@markdown ##Run this cell to check if you have GPU access
import torch

if torch.cuda.is_available():
    print("You have GPU access.")
    print(torch.cuda.get_device_name(0))
else:
    print("You do NOT have GPU access.")
    print("Go to 'Runtime -> Change runtime type' and select a GPU hardware accelerator,")
    print("then re-run the notebook. Expect slow performance on CPU.")

## **2.2. Mount your Google Drive**

In [ ]:
#@markdown ##Play the cell to connect your Google Drive to Colab
#@markdown * Follow the instructions.
#@markdown * Click on "Files" on the left. Refresh it, and your Google Drive appears as "drive".

from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
#@markdown ##Optional: use an example dataset instead of your own data
#@markdown Leave this unchecked to use your own data from Google Drive. If checked, the
#@markdown dataset is downloaded to the Colab session and its location is printed below;
#@markdown paste the folders you need into the fields in section 3.1.
Use_example_dataset = False #@param {type:"boolean"}
#@markdown Dataset name from the [CAREamics portfolio](https://github.com/CAREamics/careamics-portfolio):
example_dataset = "CARE_U2OS" #@param {type:"string"}

if Use_example_dataset:
    from careamics_portfolio import PortfolioManager

    portfolio = PortfolioManager().denoising
    if not hasattr(portfolio, example_dataset):
        raise ValueError(f"Unknown dataset. Available: {portfolio.list_datasets()}")

    example_data_path = Path("/content/example_data")
    getattr(portfolio, example_dataset).download(example_data_path)
    print(f"Downloaded to {example_data_path}. Open the Files panel on the left to see "
          f"its folder structure, then set the section 3.1 paths accordingly.")
    print("This lives in the Colab session, not your Drive, and is lost on disconnect.")


# **3. Select your parameters and paths**
---

## **3.1. Setting the main training parameters**
<font size = 4>`Training_source` (low SNR) and `Training_target` (high SNR) should each point to a folder of `.tif` or `.tiff` images, or to a single TIFF file. Paired source and target files must have **matching file names**.

<font size = 4>`model_path` is where the trained model, its checkpoints and the quality control results are written. Point it at a folder **on your Google Drive**, otherwise everything is lost when the Colab runtime shuts down.

<font size = 4>**Prediction tile size** is used in sections 5.2 and 6.1. Large images are denoised tile by tile to keep memory use low. Reduce it if you run out of GPU memory; it must be at least 128, because tiles overlap by 48 pixels.

<font size = 4>**Patch size** must be a **power of 2 and at least 8** (8, 16, 32, 64 ...) and should be smaller than the smallest dimension of your images. **Batch size** is the number of patches seen per training step; lower it if you run out of GPU memory.

<font size = 4>**Reproducibility.** Training is random in several ways: which patches are drawn from your images, how each one is augmented. By default CAREamics draws a new random seed every run, so the same data and settings give a slightly different model each time. Leave **`Use_fixed_seed`** enabled to pin the seed, which makes a run repeatable and is what you want when comparing settings. Change `seed` to see how much run-to-run variation your data produces, or disable `Use_fixed_seed` to let CAREamics pick one at random.

<font size = 4>Two caveats: on a GPU a few operations remain nondeterministic, so results can still differ in the last decimal places; and a pre-trained model loaded in section 3.3 keeps the seed stored in its own checkpoint.


In [ ]:
#@markdown ###Path to the low-SNR input images (folder of .tif/.tiff files, or a single file):
Training_source = "" #@param {type:"string"}
#@markdown ###Path to the matching high-SNR target images (folder of .tif/.tiff files, or a single file):
Training_target = "" #@param {type:"string"}

#@markdown ###Model name and output folder (use a folder on your Google Drive):
model_name = "my_care_model" #@param {type:"string"}
model_path = "" #@param {type:"string"}

#@markdown ###Training parameters
#@markdown Number of epochs:
number_of_epochs = 100 #@param {type:"number"}
#@markdown Patch size (pixels, square, a power of 2 and at least 8):
patch_size = 64 #@param {type:"number"}
#@markdown Batch size:
batch_size = 64 #@param {type:"number"}
#@markdown Number of patches held out for validation:
n_val_patches = 8 #@param {type:"number"}

#@markdown ###Prediction parameters
#@markdown Tile size used for prediction in sections 5.2 and 6.1 (pixels, square, at least 128):
prediction_tile_size = 256 #@param {type:"number"}

#@markdown ###Reproducibility
#@markdown Use a fixed random seed, so that the same data and settings reproduce the same result:
Use_fixed_seed = True #@param {type:"boolean"}
#@markdown Seed value:
seed = 42 #@param {type:"number"}


## **3.2. Data augmentation**
<font size = 4>Data augmentation (flips and 90° rotations) usually improves results and is recommended.

In [ ]:
#@markdown ##Enable or disable data augmentation:
Use_Data_augmentation = True #@param {type:"boolean"}

## **3.3. Using a pre-trained model**
<font size = 4>You can continue training from a previously trained CAREamics model. Provide the path to a checkpoint (`.ckpt`). The pre-trained model's configuration is reused, so the parameters above are ignored when this is enabled.

In [ ]:
#@markdown ##Load weights from a pre-trained CAREamics model
Use_pretrained_model = False #@param {type:"boolean"}
#@markdown ###If enabled, provide the path to the checkpoint (.ckpt) file:
pretrained_model_path = "" #@param {type:"string"}

# **4. Train the network**
---

## **4.1. Prepare the training data and model**

In [ ]:
#@markdown ##Create the configuration and the CAREamist
# Augmentations: None -> default (flips + 90-degree rotations); [] -> disabled
augmentations = None if Use_Data_augmentation else []

work_dir = require_path(model_path, "model_path")
work_dir.mkdir(parents=True, exist_ok=True)

if Use_pretrained_model:
    checkpoint_path = require_path(pretrained_model_path, "pretrained_model_path")
    print(f"Loading pre-trained model from: {checkpoint_path}")
    careamist = CAREamist(checkpoint_path=checkpoint_path, work_dir=work_dir)
else:
    config = create_advanced_care_config(
        experiment_name=model_name,
        data_type="tiff",
        axes="YX",
        patch_size=(patch_size, patch_size),
        batch_size=batch_size,
        num_epochs=number_of_epochs,
        n_val_patches=n_val_patches,
        augmentations=augmentations,
        seed=int(seed) if Use_fixed_seed else None,
    )
    print(config)
    careamist = CAREamist(config, work_dir=work_dir)


## **4.2. Start training**
<font size = 4>Training checkpoints are saved automatically to your output folder. If you loaded a pre-trained model and only want to run predictions, leave the training paths empty and skip this section.


In [ ]:
#@markdown ##Start training
training_source_text = str(Training_source).strip()
training_target_text = str(Training_target).strip()

if not training_source_text or not training_target_text:
    if Use_pretrained_model:
        print("No training paths provided. Keeping the loaded model for evaluation/prediction.")
    else:
        missing = []
        if not training_source_text:
            missing.append("Training_source")
        if not training_target_text:
            missing.append("Training_target")
        raise ValueError("Please set " + " and ".join(missing) + " before training.")
else:
    careamist.train(
        train_data=require_path(Training_source, "Training_source"),
        train_data_target=require_path(Training_target, "Training_target"),
    )
    print("Training complete.")


# **5. Evaluate your model**
---

## **5.1. Inspection of the loss function**
---
<font size = 4>**Training loss** is the error on the patches the network learned from. **Validation loss** is the same error on a small set of patches held back from training, so it shows how the network does on data it has not seen.

<font size = 4>The loss will not reach zero, because only part of the high-SNR image is predictable from the low-SNR one. Because CARE trains against a genuine high-SNR target, though, its loss falls further than the self-supervised losses of Noise2Void or Noise2Noise.

<font size = 4>These curves tell you whether training converged. Whether the model improved your images is answered in **section 5.2**.


In [ ]:
#@markdown ##Plot the training and validation loss vs. epoch
try:
    loss_dict = careamist.get_losses()
except Exception as error:
    raise RuntimeError(
        "Could not read training losses. This section is only available after "
        "training in the current work_dir."
    ) from error

if not loss_dict.get("train_loss"):
    raise RuntimeError("No training loss values were found in the CSV logs.")

plt.figure(figsize=(8, 5))
plt.plot(loss_dict["train_epoch"], loss_dict["train_loss"], label="Train loss")
if loss_dict.get("val_loss"):
    plt.plot(loss_dict["val_epoch"], loss_dict["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training losses")
plt.show()


## **5.2. Error mapping and quality metrics estimation**
---
<font size = 4>This section scores the model on a **Quality Control dataset**: pairs of low-SNR images (`Source_QC_folder`) and matching high-SNR images (`Target_QC_folder`), with **the same file name** in each folder. These fields of view should be **held back from training**.

<font size = 4>**SI-PSNR** (scale-invariant peak signal-to-noise ratio): how closely an image matches the ground truth. Higher is better; roughly +3 means the error was halved.

<font size = 4>**MicroSSIM**: structural similarity adapted to microscopy. It asks whether the same *structures* are present rather than whether pixel values match. Runs from 0 to 1, where 1 is a perfect match.

<font size = 4>The figure shows input, prediction and ground truth, with a zoomed crop of each underneath. Per-image scores and a MicroSSIM error map are written to a `Quality Control` folder next to your model, with `QC_metrics_<model_name>.csv`.


In [ ]:
#@markdown ##Provide the Quality Control folders (paired low-SNR and high-SNR .tif/.tiff images)
Source_QC_folder = "" #@param {type:"string"}
Target_QC_folder = "" #@param {type:"string"}
#@markdown Size of the zoomed-in crop shown under the full images (pixels, square):
qc_crop_size = 256 #@param {type:"number"}

source_qc_path = require_path(Source_QC_folder, "Source_QC_folder")
target_qc_path = require_path(Target_QC_folder, "Target_QC_folder")

# ZeroCostDL4Mic convention: QC results live in a "Quality Control" folder next to the
# model they describe.
qc_dir = work_dir / model_name / "Quality Control"
qc_dir.mkdir(parents=True, exist_ok=True)

# Let CAREamics list the source files itself, so that each prediction stays paired with
# the file it came from.
qc_predictions, qc_sources = careamist.predict(
    pred_data=source_qc_path,
    tile_size=(prediction_tile_size, prediction_tile_size),
)

csv_path = qc_dir / f"QC_metrics_{model_name}.csv"
scores = {"si_psnr_pred": [], "si_psnr_input": [], "ssim_pred": [], "ssim_input": []}
first_image = None  # kept for the figure; the rest are not held in memory

with open(csv_path, "w", newline="") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "image",
        "Prediction v. GT SI-PSNR",
        "Input v. GT SI-PSNR",
        "Prediction v. GT MicroSSIM",
        "Input v. GT MicroSSIM",
    ])

    for prediction, source in zip(qc_predictions, qc_sources):
        source_file = Path(source)
        target_file = target_qc_path / source_file.name
        if not target_file.exists():
            raise FileNotFoundError(
                f"No ground truth named '{source_file.name}' in {target_qc_path}. "
                "Paired QC images must have matching file names."
            )

        gt = tifffile.imread(target_file).astype(np.float32)
        noisy = tifffile.imread(source_file).astype(np.float32)
        denoised = np.asarray(prediction).squeeze().astype(np.float32)

        si_psnr_pred = scale_invariant_psnr(gt, denoised)
        si_psnr_input = scale_invariant_psnr(gt, noisy)
        ssim_pred = micro_structural_similarity(gt, denoised)
        ssim_input = micro_structural_similarity(gt, noisy)

        writer.writerow([source_file.name, si_psnr_pred, si_psnr_input,
                         ssim_pred, ssim_input])
        scores["si_psnr_pred"].append(si_psnr_pred)
        scores["si_psnr_input"].append(si_psnr_input)
        scores["ssim_pred"].append(ssim_pred)
        scores["ssim_input"].append(ssim_input)

        # Per-pixel MicroSSIM error map, saved for inspection in Fiji or napari.
        error_map = micro_structural_similarity(
            gt, denoised, return_individual_components=True
        ).SSIM.astype(np.float32)
        tifffile.imwrite(qc_dir / f"MicroSSIM_GTvsPrediction_{source_file.name}", error_map)

        if first_image is None:
            first_image = (noisy, denoised, gt, si_psnr_input, si_psnr_pred)

        print(f"{source_file.name}: SI-PSNR {si_psnr_input:.2f} -> {si_psnr_pred:.2f}, "
              f"MicroSSIM {ssim_input:.4f} -> {ssim_pred:.4f}  (input -> prediction)")

label = "Mean over QC images" if len(scores["si_psnr_pred"]) > 1 else "QC image"
print(f"\n{'':32}{'input':>10}{'prediction':>13}{'gain':>10}")
print(f"{label + ', SI-PSNR:':32}"
      f"{np.mean(scores['si_psnr_input']):>10.2f}{np.mean(scores['si_psnr_pred']):>13.2f}"
      f"{np.mean(scores['si_psnr_pred']) - np.mean(scores['si_psnr_input']):>+10.2f}")
print(f"{label + ', MicroSSIM:':32}"
      f"{np.mean(scores['ssim_input']):>10.4f}{np.mean(scores['ssim_pred']):>13.4f}"
      f"{np.mean(scores['ssim_pred']) - np.mean(scores['ssim_input']):>+10.4f}")
print(f"\nMetrics and error maps saved to: {qc_dir}")

# Display the first QC image: full field of view on top, a zoomed-in crop underneath.
noisy, denoised, gt, si_psnr_input, si_psnr_pred = first_image
height, width = gt.shape[-2:]
half = min(int(qc_crop_size), height, width) // 2
rows = slice(height // 2 - half, height // 2 + half)
cols = slice(width // 2 - half, width // 2 + half)

panels = [
    (noisy, f"Low-SNR input\nSI-PSNR: {si_psnr_input:.2f}"),
    (denoised, f"Prediction\nSI-PSNR: {si_psnr_pred:.2f}"),
    (gt, "Ground truth"),
]
fig, ax = plt.subplots(2, 3, figsize=(15, 10))
for column, (image, title) in enumerate(panels):
    ax[0, column].imshow(image, cmap="gray")
    ax[0, column].set_title(title)
    ax[1, column].imshow(image[rows, cols], cmap="gray")
for axis in ax.ravel():
    axis.axis("off")
plt.show()


# **6. Using the trained model**
---

## **6.1. Generate predictions from an unseen dataset**

In [ ]:
#@markdown ###Path to the data to denoise and the folder where results are saved:
Data_folder = "" #@param {type:"string"}
Result_folder = "" #@param {type:"string"}

data_path = require_path(Data_folder, "Data_folder")
result_dir = require_path(Result_folder, "Result_folder")
result_dir.mkdir(parents=True, exist_ok=True)

predictions, sources = careamist.predict(
    pred_data=data_path,
    tile_size=(prediction_tile_size, prediction_tile_size),
)

for pred, source in zip(predictions, sources):
    out_name = Path(source).stem + "_denoised.tif"
    out_path = result_dir / out_name
    tifffile.imwrite(out_path, np.asarray(pred).squeeze().astype(np.float32))
    print(f"Saved: {out_path}")


## **6.2. Assess the predicted output**

In [ ]:
#@markdown ##Display an input image next to its denoised prediction
idx = 0 #@param {type:"number"}
idx = int(idx)

if not predictions:
    raise RuntimeError("No predictions are available. Run section 6.1 first.")
if idx < 0 or idx >= len(predictions):
    raise IndexError(f"idx must be between 0 and {len(predictions) - 1}.")

# `sources` comes from CAREamics itself, so the input always matches the prediction.
input_img = tifffile.imread(sources[idx])
pred_img = np.asarray(predictions[idx]).squeeze()

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(input_img, cmap="gray")
ax[0].set_title("Input (low SNR)")
ax[1].imshow(pred_img, cmap="gray")
ax[1].set_title("Prediction (restored)")
for axis in ax:
    axis.axis("off")
fig.suptitle(Path(sources[idx]).name)
plt.show()


## **6.3. Download your predictions**
---

<font size = 4>**Store your data** and all its results elsewhere by downloading them from your Google Drive, and then clean up the original folder tree (dataset, results, trained model) if you plan to train or use another network. Please note that the notebook will otherwise overwrite all files which have the same name.

# **7. Version log**
---
<font size = 4>**v1.0 (CAREamics)**:
*   First release of the CAREamics-powered CARE 2D notebook. It runs alongside the existing CSBDeep `CARE_2D_ZeroCostDL4Mic` notebook, which is unchanged.
*   Built on CAREamics 0.3.2 (`create_advanced_care_config` + `CAREamist`).
*   Section 5.2 reports scale-invariant PSNR and MicroSSIM for the prediction and for the low-SNR input against the ground truth, saves MicroSSIM error maps, and writes `QC_metrics_<model_name>.csv`.
*   Optional fixed random seed (section 3.1) for reproducible training and prediction.


# **Thank you for using CARE 2D!**